In [ ]:
from google.colab import files
import pandas as pd
up = files.upload()   # choose predictions_partial.csv
PATH = next(iter(up))
df = pd.read_csv(PATH)
print('rows:', len(df), '| cols:', [c for c in df.columns if 'pred' in c or 'score' in c])
print('Amharic rows:', (df['Language']=='Amharic').sum())
print('Has OriginalText:', 'OriginalText' in df.columns)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
MODEL = 'amengemeda/amharic-hate-speech-detection-mBERT'
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
model.eval()
if torch.cuda.is_available(): model = model.cuda()
print('id2label:', model.config.id2label)

In [ ]:
# sanity check: which label is "hate"?
tests = [
    "እነዚህን ሰዎች ማጥፋት አለብን",          # "we must eliminate these people" (hateful)
    "ዛሬ ጥሩ የአየር ሁኔታ ነው",              # "the weather is nice today" (neutral)
]
for s in tests:
    enc = tok(s, return_tensors='pt', truncation=True, max_length=128)
    if torch.cuda.is_available(): enc = {k:v.cuda() for k,v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    pred = int(logits.argmax(-1))
    print(f"pred={pred}  | {s}")

In [ ]:
hate_ids = {1}   # confirmed: label 1 = hate
MAXLEN = 128
labels, preds = [], []
amh_mask = df['Language'] == 'Amharic'
rows = df[amh_mask]
for n, (i, row) in enumerate(rows.iterrows()):
    enc = tok(str(row['OriginalText'])[:2000], truncation=True, max_length=MAXLEN, return_tensors='pt')
    if torch.cuda.is_available(): enc = {k: v.cuda() for k, v in enc.items()}
    with torch.no_grad():
        idx = int(model(**enc).logits.argmax(-1))
    labels.append(idx); preds.append(int(idx in hate_ids))
    if (n+1) % 100 == 0: print(f'  mBERT {n+1}/{len(rows)}')
df['ammbert_label'] = None
df['ammbert_pred'] = None
df.loc[amh_mask, 'ammbert_label'] = labels
df.loc[amh_mask, 'ammbert_pred'] = preds
print('Amharic mBERT Hate predictions:', sum(preds), 'of', len(preds), 'Amharic rows')

In [ ]:
df.to_csv('predictions_full.csv', index=False)
from google.colab import files
files.download('predictions_full.csv')
print('Saved predictions_full.csv with:',
      [c for c in df.columns if 'pred' in c or 'score' in c])

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import pandas as pd
if 'gold_bin' not in df.columns:
    df['gold_bin'] = (df['Label3Class'].astype(str).str.strip()=='Hate').astype(int)
CLF = [('Perspective','persp_pred'), ('AfriHate','afrihate_pred'), ('Amharic mBERT','ammbert_pred')]
def mrow(name, scope, frame, col):
    s = frame[frame[col].notna()]
    if len(s)==0: return None
    yt=s['gold_bin'].astype(int); yp=s[col].astype(int)
    tn,fp,fn,tp = confusion_matrix(yt,yp,labels=[0,1]).ravel()
    return {'classifier':name,'scope':scope,
            'coverage':f'{len(s)}/{len(frame)} ({len(s)/len(frame):.0%})',
            'accuracy':round(accuracy_score(yt,yp),3),'precision':round(precision_score(yt,yp,zero_division=0),3),
            'recall':round(recall_score(yt,yp,zero_division=0),3),'f1':round(f1_score(yt,yp,zero_division=0),3),
            'TP':tp,'FP':fp,'FN':fn,'TN':tn}
rows=[]
for name,col in CLF:
    if col not in df.columns: continue
    rows.append(mrow(name,'Overall',df,col))
    for lang in ['Amharic','Afan Oromo']:
        r=mrow(name,lang,df[df['Language']==lang],col)
        if r: rows.append(r)
res=pd.DataFrame([r for r in rows if r])
print(res.to_string(index=False))
res.to_csv('stage2_results_full.csv', index=False)
files.download('stage2_results_full.csv')

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import pandas as pd
if 'gold_bin' not in df.columns:
    df['gold_bin'] = (df['Label3Class'].astype(str).str.strip()=='Hate').astype(int)
CLF = [('Perspective','persp_pred'), ('AfriHate','afrihate_pred'), ('Amharic mBERT','ammbert_pred')]
def mrow(name, scope, frame, col):
    s = frame[frame[col].notna()]
    if len(s)==0: return None
    yt=s['gold_bin'].astype(int); yp=s[col].astype(int)
    tn,fp,fn,tp = confusion_matrix(yt,yp,labels=[0,1]).ravel()
    return {'classifier':name,'scope':scope,
            'coverage':f'{len(s)}/{len(frame)} ({len(s)/len(frame):.0%})',
            'accuracy':round(accuracy_score(yt,yp),3),'precision':round(precision_score(yt,yp,zero_division=0),3),
            'recall':round(recall_score(yt,yp,zero_division=0),3),'f1':round(f1_score(yt,yp,zero_division=0),3),
            'TP':tp,'FP':fp,'FN':fn,'TN':tn}
rows=[]
for name,col in CLF:
    if col not in df.columns: continue
    rows.append(mrow(name,'Overall',df,col))
    for lang in ['Amharic','Afan Oromo']:
        r=mrow(name,lang,df[df['Language']==lang],col)
        if r: rows.append(r)
res=pd.DataFrame([r for r in rows if r])
print(res.to_string(index=False))
res.to_csv('stage2_results_full.csv', index=False)
from google.colab import files
files.download('stage2_results_full.csv')